# Module 05 - Research ideas: proofs of concept

Homework: [homework5.md](https://github.com/DataTalksClub/stock-markets-analytics-zoomcamp/blob/main/cohorts/2026/homework5.md) ·
Submit: [hw05](https://courses.datatalks.club/sma-zoomcamp-2026/homework/hw05) · Write-up: [answers.md](answers.md)

Kernel: **Python (stock-markets-zoomcamp)**.

One small test per idea, using **only free data**. Each POC checks that the pipeline works and
whether the first signs support the hypothesis. None of them is the final test. Each cell says which free proxy
stands in for the data the full test needs. See *Data plan* at the end.

| # | Idea | Free data used here | Free / low-cost source for the full test |
| --- | --- | --- | --- |
| 1 | SET100 daily ML + foreign flows | yfinance `.BK`; USD/THB, EEM and VIX as foreign-pressure proxies | SET investor-type and NVDR reports, SETSMART / CMRI via the university |
| 2 | SET50 options: vol forecasts + cost-aware hedging | TDEX (SET50 ETF); simulated option book | TFEX series pages, TFEX data request |
| 3 | SPX variance-risk-premium timing | Cboe PUT / PUTY / WPUT / CNDR indices, VIX family, FRED T-bill | Same, plus OptionsDX chains if needed |

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import norm
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

from smaz import data, utils

import warnings
from scipy.stats import ConstantInputWarning
warnings.filterwarnings("ignore", category=ConstantInputWarning)     # days where a model's scores are all equal
warnings.filterwarnings("ignore", message="divide by zero encountered in log")

utils.set_plot_defaults()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:.4f}".format)


def wide(df, field="Close"):
    """Long yfinance frame -> Date x Ticker."""
    return df.pivot_table(index="Date", columns="Ticker", values=field)


def lagged_to(index, frame):
    """US / FX closes print after the Thai close, so only the previous day's value is known at the Thai close."""
    out = frame.shift(1).reindex(frame.index.union(index)).ffill().reindex(index)
    out.index.name = "Date"
    return out


def t_stat(x):
    x = pd.Series(x).dropna()
    return x.mean() / x.std() * np.sqrt(len(x))

---
## Idea 1 - SET100 daily ML with foreign-flow features

**H1:** adding foreign-flow features improves an out-of-sample daily forecast of next-day relative returns.
**H2:** the edge is *reversal after foreign-driven selling* (price pressure). If so, the losers' rebound is larger in
risk-off regimes (high VIX, weakening baht) and in the liquid names foreigners trade.

**POC data.** yfinance `.BK` prices for current SET100 names. Market-level proxies, all lagged one day:
USD/THB (baht weakening = outflow), EEM (EM fund flows), VIX (risk-off). A stock's beta to EEM stands in for how
much foreigners trade it.

**Known biases, to fix later:** today's membership (survivorship), no delisted names, no NVDR data. Membership can be rebuilt from SET's semi-annual announcements; delisted prices and NVDR data via SETSMART / CMRI.

In [ ]:
SET100 = '''ADVANC AEONTS AMATA AOT AP AWC BAM BANPU BBL BCH BCP BCPG BDMS BEM BGRIM BH BJC BLA BTG BTS
CBG CCET CENTEL CHG CK CKP COM7 CPALL CPF CPN CRC DELTA DOHOME EA EGCO ERW GFPT GLOBAL GPSC GULF GUNKUL HANA
HMPRO ICHI IRPC ITC IVL JMART JMT KBANK KCE KKP KTB KTC LH M MEGA MINT MTC OR ORI OSP PLANB PR9 PTT PTTEP
PTTGC QH RATCH RCL SAWAD SCB SCC SCGP SIRI SISB SJWD SPALI SPRC STA STGT TASCO TCAP THANI TIDLOR TISCO TOA
TOP TRUE TTB TU TTW VGI WHA WHAUP BA AAV'''.split()

raw = data.load_ohlcv([f"{t}.BK" for t in SET100], start="2012-01-01")
close, volume = wide(raw, "Close"), wide(raw, "Volume")
close = close.loc[:, close.notna().sum() >= 750]          # at least ~3 years of history
volume = volume[close.columns]

ret = close.pct_change(fill_method=None)
ret = ret.where(ret.abs() < 0.35)                          # SET daily limit is +-30%; bigger = bad print / corporate action
dollar_vol = (close * volume).replace(0, np.nan)

mac = wide(data.load_ohlcv(["THB=X", "EEM", "^VIX"], start="2011-06-01"))
mac_ret = mac.pct_change(fill_method=None)
macro = lagged_to(close.index, pd.DataFrame({
    "thb_ret_lag": mac_ret["THB=X"],    # > 0 = baht weakened = foreign outflow proxy
    "eem_ret_lag": mac_ret["EEM"],
    "vix_lag": mac["^VIX"],
}))
print(f"{close.shape[1]} stocks, {close.index.min().date()} -> {close.index.max().date()}")

In [ ]:
eem = macro["eem_ret_lag"]
stock_feats = {
    "r1": ret,
    "r5": close.pct_change(5, fill_method=None),
    "r20": close.pct_change(20, fill_method=None),
    "vol20": ret.rolling(20).std(),
    "liq": np.log(dollar_vol.rolling(60).mean()),
    "dv_shock": np.log(dollar_vol) - np.log(dollar_vol).rolling(20).mean(),
    "dist_hi": close / close.rolling(250).max() - 1,
    "beta_eem": ret.apply(lambda s: s.rolling(120).cov(eem)).div(eem.rolling(120).var(), axis=0),
}
fwd = ret.shift(-1)
target = fwd.sub(fwd.mean(axis=1), axis=0)                 # next-day return relative to the equal-weight universe

panel = pd.concat({k: v.stack(future_stack=True) for k, v in {**stock_feats, "y": target, "fwd_ret": fwd}.items()}, axis=1)
panel.index.names = ["Date", "Ticker"]
STOCK = list(stock_feats)
panel[STOCK] = panel.groupby(level="Date")[STOCK].rank(pct=True)   # cross-sectional ranks: robust to outliers
panel = panel.join(macro, on="Date")
panel["r1_x_thb"] = (panel["r1"] - 0.5) * panel["thb_ret_lag"]
panel["r1_x_vix"] = (panel["r1"] - 0.5) * panel["vix_lag"]

BASE = ["r1", "r5", "r20", "vol20", "liq", "dv_shock", "dist_hi"]
FLOW = BASE + ["beta_eem", "thb_ret_lag", "eem_ret_lag", "vix_lag", "r1_x_thb", "r1_x_vix"]
panel = panel.dropna(subset=FLOW + ["y"])
print(f"panel rows: {len(panel):,}")

In [ ]:
def walk_forward(df, cols, target="y", years=range(2017, 2027), embargo=5, **xgb):
    """Refit once a year on all earlier data, leaving an `embargo`-day gap before the test year."""
    params = dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                  min_child_weight=50, tree_method="hist", n_jobs=-1, random_state=0) | xgb
    dates = df.index.get_level_values("Date")
    uniq = np.sort(dates.unique())
    preds = []
    for yr in years:
        test = dates.year == yr
        if not test.any():
            continue
        cutoff = uniq[max(np.searchsorted(uniq, dates[test].min()) - embargo, 0)]
        train = dates < cutoff
        model = XGBRegressor(**params).fit(df.loc[train, cols], df.loc[train, target])
        preds.append(pd.Series(model.predict(df.loc[test, cols]), index=df.index[test]))
    return pd.concat(preds)


def evaluate(pred, cost_per_side=0.0015):
    """IC + a daily-rebalanced top-decile long book vs the equal-weight universe."""
    df = panel.loc[pred.index, ["y", "fwd_ret"]].assign(pred=pred)
    ic = df.groupby(level="Date").apply(lambda g: g["pred"].corr(g["y"], method="spearman"))
    top = (df.groupby(level="Date")["pred"].rank(pct=True) >= 0.9).astype(float)
    w = top / top.groupby(level="Date").transform("sum")
    excess = (w * df["fwd_ret"]).groupby(level="Date").sum() - df["fwd_ret"].groupby(level="Date").mean()
    turnover = w.unstack().fillna(0).diff().abs().sum(axis=1)
    net = excess - turnover * cost_per_side
    return pd.Series({
        "IC mean": ic.mean(), "IC t-stat": t_stat(ic),
        "excess % p.a. (gross)": excess.mean() * 252 * 100, "Sharpe (gross)": excess.mean() / excess.std() * np.sqrt(252),
        "turnover / day": turnover.mean(), "excess % p.a. (net)": net.mean() * 252 * 100,
    })


pred_base = walk_forward(panel, BASE)
pred_flow = walk_forward(panel, FLOW)
q1_table = pd.DataFrame({"baseline (price/volume)": evaluate(pred_base), "+ flow proxies": evaluate(pred_flow)})
q1_table

In [ ]:
# H1 check: is the flow model's daily IC better than the baseline's on the same days? (paired t-test on IC difference)
def daily_ic(pred):
    df = panel.loc[pred.index, ["y"]].assign(pred=pred)
    return df.groupby(level="Date").apply(lambda g: g["pred"].corr(g["y"], method="spearman"))

ic_diff = daily_ic(pred_flow) - daily_ic(pred_base)
print(f"IC(flow) - IC(base): mean {ic_diff.mean():.4f}, t-stat {t_stat(ic_diff):.2f}, days {len(ic_diff)}")

In [ ]:
# H2 mechanism (no ML): next-day return of today's biggest losers, relative to the universe, by regime
losers = panel[panel["r1"] <= 0.1]
rebound = losers.groupby(level="Date")["y"].mean()
reg = macro.reindex(rebound.index)
vix_hi = reg["vix_lag"] > reg["vix_lag"].rolling(250, min_periods=60).median()   # vs trailing median: no look-ahead
thb_weak = reg["thb_ret_lag"] > 0.003
liq_hi = losers.groupby(level="Date").apply(lambda g: g.loc[g["liq"] >= 2 / 3, "y"].mean())
liq_lo = losers.groupby(level="Date").apply(lambda g: g.loc[g["liq"] <= 1 / 3, "y"].mean())

rows = {
    "all days": rebound, "VIX above trailing median": rebound[vix_hi], "VIX below": rebound[~vix_hi],
    "baht weakened > 0.3% (outflow proxy)": rebound[thb_weak], "other days": rebound[~thb_weak],
    "losers in most-liquid third": liq_hi, "losers in least-liquid third": liq_lo,
}
q1_h2 = pd.DataFrame({k: {"bps / day": v.mean() * 1e4, "t-stat": t_stat(v), "days": v.count()} for k, v in rows.items()}).T
q1_h2

---
## Idea 2 - SET50 options: ML volatility forecasts and cost-aware hedging

Two parts that can be tested without TFEX option quotes:

* **2a, vol forecast (input to the vol-gap signal).** Can ML with Thai and global features forecast SET50 20-day
  realized vol better than HAR-RV? The real signal is then forecast − TFEX ATM IV, from TFEX settlement prices.
* **2b, hedging (H2 of B1).** Does a cost-aware no-trade band beat plain BS delta hedging by *more* as costs rise?
  Simulated book: short 10 SET50 calls, hedged in **whole SET50 futures contracts**. Both contracts are 200 THB per
  index point, so the hedge is lumpy. The band rule is the analytic version of the no-transaction-band network
  (Imaki et al., 2021). A deep version needs `uv sync --group dl` (torch).

In [ ]:
etf = data.load_ohlcv(["TDEX.BK"], start="2008-01-01").set_index("Date")   # ThaiDEX SET50 ETF tracks the SET50
lr = np.log(etf["Close"]).diff()
ann = lambda var: np.sqrt(252 * var)

vol = pd.DataFrame(index=etf.index)
vol["y"] = np.log(ann((lr**2).rolling(21).mean().shift(-21)))           # log realized vol over the NEXT 21 days
vol["rv1"] = np.log(ann(lr**2) + 1e-3)
vol["rv5"] = np.log(ann((lr**2).rolling(5).mean()))
vol["rv22"] = np.log(ann((lr**2).rolling(22).mean()))
park = np.log(etf["High"] / etf["Low"]) ** 2 / (4 * np.log(2))           # Parkinson range variance
vol["park22"] = np.log(ann(park.rolling(22).mean()) + 1e-3)
vol["ret22"] = etf["Close"].pct_change(22)

mac2 = wide(data.load_ohlcv(["THB=X", "EEM", "^VIX"], start="2007-06-01"))
m2 = lagged_to(vol.index, pd.DataFrame({
    "log_vix": np.log(mac2["^VIX"]),
    "thb_vol22": mac2["THB=X"].pct_change(fill_method=None).rolling(22).std() * np.sqrt(252),
    "eem_vol22": mac2["EEM"].pct_change(fill_method=None).rolling(22).std() * np.sqrt(252),
}))
vol = vol.join(m2).replace([np.inf, -np.inf], np.nan).dropna()
HAR = ["rv1", "rv5", "rv22"]
ML = HAR + ["park22", "ret22", "log_vix", "thb_vol22", "eem_vol22"]
print(f"{len(vol):,} days, {vol.index.min().date()} -> {vol.index.max().date()}")

In [ ]:
preds = {"random walk (rv22)": [], "HAR-RV (OLS)": [], "HAR-X (OLS, + Thai/global)": [], "XGBoost (HAR + Thai/global)": []}
truth = []
for yr in range(2013, 2027):
    test = vol.index.year == yr
    if not test.any():
        continue
    train = vol.index < vol.index[test][0] - pd.Timedelta(days=35)       # gap > 21 trading days: targets overlap
    tr, te = vol[train], vol[test]
    preds["random walk (rv22)"].append(te["rv22"])
    preds["HAR-RV (OLS)"].append(pd.Series(LinearRegression().fit(tr[HAR], tr["y"]).predict(te[HAR]), te.index))
    preds["HAR-X (OLS, + Thai/global)"].append(pd.Series(LinearRegression().fit(tr[ML], tr["y"]).predict(te[ML]), te.index))
    xgb = XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.03, subsample=0.8, min_child_weight=20,
                       random_state=0).fit(tr[ML], tr["y"])
    preds["XGBoost (HAR + Thai/global)"].append(pd.Series(xgb.predict(te[ML]), te.index))
    truth.append(te["y"])

truth = pd.concat(truth)
preds = {k: pd.concat(v) for k, v in preds.items()}

def qlike(y_log, f_log):        # variance-based QLIKE loss, robust to noisy vol proxies (Patton, 2011)
    r = np.exp(2 * y_log) / np.exp(2 * f_log)
    return (r - np.log(r) - 1).mean()

q2a = pd.DataFrame({k: {"RMSE (log vol)": np.sqrt(((truth - p) ** 2).mean()), "QLIKE": qlike(truth, p)}
                    for k, p in preds.items()}).T
for rival in ["HAR-X (OLS, + Thai/global)", "XGBoost (HAR + Thai/global)"]:
    d = (truth - preds["HAR-RV (OLS)"]) ** 2 - (truth - preds[rival]) ** 2
    dm = sm.OLS(d.values, np.ones(len(d))).fit(cov_type="HAC", cov_kwds={"maxlags": 21})
    print(f"Diebold-Mariano HAR vs {rival}: loss diff {dm.params[0]:+.4f}, t {dm.tvalues[0]:+.2f}  (> 0 favours {rival.split(' ')[0]})")
q2a

In [ ]:
rng = np.random.default_rng(42)
SIGMA = float(np.sqrt(252) * lr.dropna().std())     # SET50 vol from TDEX history
S0 = K = 1000.0
STEPS, MULT = 21, 200                               # 1-month ATM call, THB 200 per index point


def bs_call(S, tau, sigma=SIGMA):
    d1 = (np.log(S / K) + 0.5 * sigma**2 * tau) / (sigma * np.sqrt(tau))
    return S * norm.cdf(d1) - K * norm.cdf(d1 - sigma * np.sqrt(tau)), norm.cdf(d1)


def paths(n, kind):
    if kind == "GBM":
        steps = -0.5 * SIGMA**2 / 252 + SIGMA / np.sqrt(252) * rng.standard_normal((n, STEPS))
    else:                                            # 5-day block bootstrap of real TDEX returns: fat tails, vol clusters
        hist = (lr.dropna() - lr.mean()).values
        starts = rng.integers(0, len(hist) - 5, size=(n, -(-STEPS // 5)))
        steps = hist[starts[:, :, None] + np.arange(5)].reshape(n, -1)[:, :STEPS]
    return S0 * np.exp(np.hstack([np.zeros((n, 1)), np.cumsum(steps, axis=1)]))


def hedge_pnl(S, cost_bps, band, n_opt):
    """P&L (THB) of selling n_opt calls at BS value and hedging in whole futures contracts."""
    premium, _ = bs_call(S0, STEPS / 252)
    pnl = np.full(len(S), premium * n_opt * MULT)
    h = np.zeros(len(S))
    for t in range(STEPS):
        _, delta = bs_call(S[:, t], (STEPS - t) / 252)
        target = n_opt * delta
        gap = target - h
        new = np.round(target) if band == 0 else np.where(np.abs(gap) > band, np.round(target - np.sign(gap) * band), h)
        pnl -= np.abs(new - h) * S[:, t] * MULT * cost_bps / 1e4
        h = new
        pnl += h * (S[:, t + 1] - S[:, t]) * MULT
    return pnl - n_opt * MULT * np.maximum(S[:, -1] - K, 0)


cvar = lambda pnl, a=0.05: -np.sort(pnl)[: int(a * len(pnl))].mean()      # expected loss in the worst 5%
rows = []
for kind in ["GBM", "bootstrap"]:
    train_S, test_S = paths(20_000, kind), paths(20_000, kind)
    for n_opt, bands in [(10, [0, 0.5, 1, 1.5, 2, 3]), (100, [0, 0.5, 1, 2, 3, 4, 6, 8])]:   # band in futures contracts
        for cost in [0, 2, 5, 10, 20, 40]:
            best = min(bands, key=lambda b: cvar(hedge_pnl(train_S, cost, b, n_opt)))   # chosen on training paths only
            c_delta, c_band = cvar(hedge_pnl(test_S, cost, 0, n_opt)), cvar(hedge_pnl(test_S, cost, best, n_opt))
            rows.append({"paths": kind, "book (calls)": n_opt, "cost (bps/side)": cost,
                         "CVaR95 BS delta (THB)": c_delta, "CVaR95 band (THB)": c_band, "best band": best,
                         "risk cut %": 100 * (1 - c_band / c_delta)})
q2b = pd.DataFrame(rows)
print(f"SET50 vol used: {SIGMA:.1%}  | premium per call: THB {bs_call(S0, STEPS / 252)[0] * MULT:,.0f}")
print("risk cut % = how much the cost-aware band lowers CVaR95 vs BS delta, on out-of-sample paths")
q2b.pivot_table(index="cost (bps/side)", columns=["paths", "book (calls)"], values="risk cut %")

---
## Idea 3 - Timing the S&P 500 variance risk premium

**H1:** a model that knows the state of the vol market (VRP, VIX term structure, VVIX, SKEW) earns a higher Sharpe
and a smaller drawdown than selling puts every month.

**POC data (all free).** Cboe publishes daily histories of strategy indices that sell SPX options with collateral held
in T-bills, so no option chains are needed:
**PUT** (ATM monthly put-write), **PUTY** (2% OTM), **WPUT** (weekly) and **CNDR** (monthly iron condor, the
defined-risk version closest to the put spreads a retail account can trade). When a rule is out, it earns T-bills.
Training starts in 2007. Test years run 2014 onward, refit every January.

In [ ]:
def load_cboe(symbol):
    """Daily history of a Cboe index from Cboe's free CSV feed, cached to parquet."""
    def _fetch():
        df = pd.read_csv(f"https://cdn.cboe.com/api/global/us_indices/daily_prices/{symbol}_History.csv")
        return pd.DataFrame({"Close": df.iloc[:, -1].values}, index=pd.to_datetime(df["DATE"], format="%m/%d/%Y"))
    return data.cached("cboe", _fetch, symbol=symbol)["Close"]


VARIANTS = {"PUT": "ATM monthly put-write", "PUTY": "2% OTM monthly put-write",
            "WPUT": "ATM weekly put-write", "CNDR": "monthly iron condor (defined risk)"}

d3 = wide(data.load_ohlcv(["^GSPC", "^VIX", "^VIX3M", "^VVIX", "^SKEW", "SPY"], start="2006-01-01"))
tbill = data.load_fred("DTB3", start="2006-01-01").set_index("DATE")["DTB3"].ffill()
cboe_m = pd.DataFrame({s: load_cboe(s) for s in VARIANTS}).resample("ME").last()

spx_lr = np.log(d3["^GSPC"]).diff()
daily = pd.DataFrame({
    "vix": d3["^VIX"],
    "vrp": (d3["^VIX"] / 100) ** 2 - 252 * (spx_lr**2).rolling(22).mean(),   # implied minus trailing realized variance
    "slope": d3["^VIX"] / d3["^VIX3M"],                                       # > 1 = backwardation (stress)
    "vvix": d3["^VVIX"], "skew": d3["^SKEW"],
    "spx_1m": d3["^GSPC"].pct_change(21), "spx_3m": d3["^GSPC"].pct_change(63),
    "rv22": np.sqrt(252 * (spx_lr**2).rolling(22).mean()),
})
m = daily.resample("ME").last()
for s in VARIANTS:
    m[f"{s}_next"] = cboe_m[s].pct_change().shift(-1)
m["spy_next"] = d3["SPY"].resample("ME").last().pct_change().shift(-1)
m["tb_next"] = tbill.resample("ME").last() / 100 / 12
m = m.dropna()
FEATS3 = ["vix", "vrp", "slope", "vvix", "skew", "spx_1m", "spx_3m", "rv22"]
print(f"{len(m)} months, {m.index.min().date()} -> {m.index.max().date()}")

In [ ]:
def timing_preds(ret_col):
    """Walk-forward XGBoost forecast of next month's excess return of one strategy index."""
    preds = []
    for yr in range(2014, 2027):
        test = m.index.year == yr
        if not test.any():
            continue
        train = m.index < m.index[test][0]
        model = XGBRegressor(n_estimators=200, max_depth=2, learning_rate=0.03, subsample=0.8, min_child_weight=10,
                             random_state=0).fit(m.loc[train, FEATS3], m.loc[train, ret_col] - m.loc[train, "tb_next"])
        preds.append(pd.Series(model.predict(m.loc[test, FEATS3]), m.index[test]))
    return pd.concat(preds)


pred3 = timing_preds("PUT_next")
t3 = m.loc[pred3.index]
in_ml = pred3 > 0
in_lin = t3["slope"] < 1                                    # simple rule: sell only in contango
strategies = {
    "SPY buy & hold": t3["spy_next"],
    "PUT index (always sell)": t3["PUT_next"],
    "linear rule: VIX < VIX3M": np.where(in_lin, t3["PUT_next"], t3["tb_next"]),
    "XGBoost timing": np.where(in_ml, t3["PUT_next"], t3["tb_next"]),
}


def stats(r, invested=None):
    r = pd.Series(r, index=t3.index)
    ex = r - t3["tb_next"]
    curve = (1 + r).cumprod()
    return {"CAGR %": 100 * (curve.iloc[-1] ** (12 / len(r)) - 1), "vol %": 100 * r.std() * np.sqrt(12),
            "Sharpe": ex.mean() / ex.std() * np.sqrt(12), "max DD %": 100 * (curve / curve.cummax() - 1).min(),
            "worst month %": 100 * r.min(), "months invested %": 100 * (1 if invested is None else invested.mean())}


q3 = pd.DataFrame({k: stats(v, inv) for (k, v), inv in
                   zip(strategies.items(), [None, None, in_lin, in_ml])}).T
print(f"test window: {t3.index.min():%Y-%m} -> {t3.index.max():%Y-%m} ({len(t3)} months)")
q3

In [ ]:
# Is the timing gain more than luck? 6-month block bootstrap of the test months
def sharpe(ex):
    return ex.mean() / ex.std() * np.sqrt(12)

ex = pd.DataFrame({k: pd.Series(v, index=t3.index) - t3["tb_next"] for k, v in strategies.items()})
rng3 = np.random.default_rng(7)
n, block = len(ex), 6
wins = {"XGBoost timing": 0, "linear rule: VIX < VIX3M": 0}
for _ in range(2000):
    idx = np.concatenate([np.arange(s, s + block) % n for s in rng3.integers(0, n, n // block + 1)])[:n]
    sample = ex.iloc[idx]
    for k in wins:
        wins[k] += sharpe(sample[k]) > sharpe(sample["PUT index (always sell)"])
for k, w in wins.items():
    print(f"P(Sharpe {k} > always-sell PUT) = {w / 2000:.2f}")

In [ ]:
# Same two rules on the other free Cboe strategy indices: OTM, weekly, and defined-risk (iron condor)
rows = {}
for s, label in VARIANTS.items():
    p = timing_preds(f"{s}_next")
    r = t3[f"{s}_next"]
    for rule, inv in {"always sell": None, "VIX < VIX3M": in_lin, "XGBoost": p > 0}.items():
        st = stats(r if inv is None else np.where(inv, r, t3["tb_next"]), inv)
        rows[(label, rule)] = {k: st[k] for k in ["CAGR %", "Sharpe", "max DD %", "worst month %", "months invested %"]}
q3_variants = pd.DataFrame(rows).T
q3_variants

---
## Data plan

Every POC above runs on free data. For the full tests, these are the free / university-library sources (checked = I downloaded or opened it).

| Idea | Data needed | Free / low-cost source | How | Status |
| --- | --- | --- | --- | --- |
| 1 | Daily prices of SET / mai stocks | yfinance `.BK` | `smaz.data.load_ohlcv` | checked |
| 1 | Point-in-time SET50 / SET100 membership | SET's semi-annual constituent announcements (set.or.th news) | Rebuild by hand: 2 lists a year, about 30 since 2012 | verify |
| 1 | Market-level net flows by investor type (foreign / institution / prop / retail) | SET website "trading by investor type"; TFEX "trading by investor types" | Manual download (SET blocks scripts: HTTP 403) | page checked |
| 1 | Stock-level NVDR trading | SET / Thai NVDR daily reports (public top-30 lists); full cross-section via SETSMART (university library) or a CMRI data request | Manual / request | verify |
| 2 | SET50 level and realized vol | TDEX ETF on yfinance (Yahoo has no SET50 index history) | `load_ohlcv("TDEX.BK")` | checked |
| 2 | SET50 option settlement prices / IV by series | tfex.co.th series pages (recent data); longer history by TFEX or SETSMART request | Collect daily from now; request history | verify |
| 3 | VIX, VIX3M, VVIX, SKEW, PUT, PUTY, WPUT, CNDR | Cboe free CSV feed + yfinance | `load_cboe(...)` | checked |
| 3 | 3-month T-bill | FRED `DTB3` | `smaz.data.load_fred` | checked |
| 3 | SPX / XSP option chains by strike (only if the index proxies are not enough) | OptionsDX (paid, about USD 50 per year of data), or snapshot yfinance chains daily from now | Buy one year to start | verify |

---
## Summary of POC results

Free data only, run on 1 Nov 2026. These are first signs, not the final tests.

| Idea | Hypothesis | POC result | Verdict so far | What the full test needs (free / low-cost) |
| --- | --- | --- | --- | --- |
| 1 | H1: flow features improve the daily forecast | IC 0.057 -> 0.063 with market-level flow proxies; difference t = 1.39 | **Not shown** with proxies | Stock-level NVDR / foreign flow (SET reports, SETSMART / CMRI) |
| 1 | Daily ML edge survives costs | Gross top-decile excess 28-32% p.a. (Sharpe ~2), but turnover 130% of the book per day -> **net -19% p.a.** at 0.15%/side | **Fails as built** | Hold 3-5 days, trade only strong signals, cost-aware portfolio |
| 1 | H2: rebound = liquidity provision | Losers rebound 11.7 bps/day (t 6.5); larger in illiquid third (15.3 vs 9.1) and on baht-weak days (14.4 vs 11.1); **not** larger when VIX is high (10.2 vs 13.1) | Mixed: the illiquid-stock result also fits bid-ask bounce | Open-to-close returns (yfinance has opens) to remove the bounce |
| 2a | ML beats HAR for SET50 vol | HAR best (RMSE 0.371); HAR + Thai/global features 0.384 (DM t -2.33); XGBoost 0.439 (DM t -3.15) | **Rejected** on daily data | TFEX implied vol as an input; intraday data if obtainable |
| 2b | Cost-aware hedging gains grow with costs | Risk cut vs BS delta, 100-call book: 1.8% at 2 bps -> 4.8% at 10 -> 15.8% at 40 (GBM); 0-4.8% on bootstrapped real returns; a 10-call book gains nothing below 20 bps because whole-contract rounding already acts as a band | **Supported in direction**; small at retail cost levels | TFEX fee schedule + settlement prices; a deep (torch) no-transaction-band net |
| 3 | ML timing beats always selling | XGBoost Sharpe 0.28 vs 0.70 for PUT (P(better) = 0.01); VIX < VIX3M rule: Sharpe 0.80, max DD -15% vs -21%, worst month -7.6% vs -13.4%, P(better Sharpe) = 0.56 | ML **rejected**; the simple rule cuts tail risk | - |
| 3 | Which way of selling suits a retail account | 2% OTM put-write + VIX < VIX3M: Sharpe 0.66 -> 0.90, max DD -20% -> -10%. Weekly put-write: Sharpe ~0.23, no help from timing. Iron condor (defined risk): CAGR ~1%, Sharpe below 0, so the protective wings cost most of the premium | OTM monthly selling with the term-structure filter is the best free-data candidate | Option chains (OptionsDX) to test real spread widths and costs |

**Takeaways**

1. All three POCs run on free data. For the full tests, the data gaps are Thai flow data (idea 1) and TFEX option
   history (idea 2). Both can come from SET / TFEX / SETSMART / CMRI through the university.
2. In both idea 2a and idea 3, simple economic models (HAR, the term-structure rule) beat tree models when data is
   monthly or daily and small. Any ML gain has to come from better data, not a more complex model.
3. For personal trading (idea 3): selling ~2% OTM monthly puts only when VIX < VIX3M halved the drawdown. Cheap
   protection (iron condor) gave up nearly all the premium. Paper-trade it before using real money.
